# Part 2: Build a Summary Agent Using Wikipedia Pages

Your task in Part 2 is to implement an agent that can perform three core functions:

1. Fetch Web Page – The agent should be able to fetch the content of a web page given its URL.

2. Save Summary – The agent should be able to save a summary of the page it processed.

3. Search – The agent should be able to perform a search for relevant or related information.


## Question 4. Framework and LLM Provider

- You can use any framework (ToyAIKit, Agents SDK, Pydantic AI, LangChain, LlamaIndex, etc.) or implement these functions yourself from scratch.

- You can also use any LLM provider of your choice (OpenAI, Anthropic, Mistral, Azure, or others).

- In your answer field, please write:

    - Which framework you chose (if any)

    - Which LLM provider you used

- If you don't know what to choose, you can go with Pydantic AI (my favorite) and OpenAI (we use it in the course).

- If you have time, I suggest trying to implement some of the things yourself before using a framework. You can refer to lectures and ToyAIKit code for details.

In [1]:
from typing import Iterable, Callable, Any, Dict, List

def sliding_window(
        seq: Iterable[Any],
        size: int,
        step: int
    ) -> List[Dict[str, Any]]:
    """
    Create overlapping chunks from a sequence using a sliding window approach.

    Args:
        seq: The input sequence (string or list) to be chunked.
        size (int): The size of each chunk/window.
        step (int): The step size between consecutive windows.

    Returns:
        lifst: A list of dictionaries, each containing:
            - 'start': The starting position of the chunk in the original sequence
            - 'content': The chunk content

    Raises:
        ValueError: If size or step are not positive integers.

    Example:
        >>> sliding_window("hello world", size=5, step=3)
        [{'start': 0, 'content': 'hello'}, {'start': 3, 'content': 'lo wo'}]
    """
    if size <= 0 or step <= 0:
        raise ValueError("size and step must be positive")

    n = len(seq)
    result = []
    for i in range(0, n, step):
        batch = seq[i:i+size]
        result.append({'start': i, 'content': batch})
        if i + size > n:
            break
    return result

In [2]:
# Fetch a webpage
import requests
from typing import Optional

reader_url_prefix = "https://r.jina.ai/"

def get_page_content(url: str) -> Optional[str]:
    """
    Fetch the Markdown content of a web page using the Jina Reader service.

    This function prepends the Jina Reader proxy URL to the provided `url`,
    sends a GET request with a timeout, and decodes the response as UTF-8 text.

    Args:
        url (str): The URL of the page to fetch.

    Returns:
        Optional[str]: The Markdown-formatted content of the page if the request
        succeeds; otherwise, None.

    Raises:
        None: All network or decoding errors are caught and suppressed.
               Logs or error messages could be added as needed.
    """
    reader_url = reader_url_prefix + url

    try:
        response = requests.get(reader_url, timeout=10)
        response.raise_for_status()  # raises for 4xx/5xx HTTP errors
        return response.content.decode("utf-8")
    except (requests.exceptions.RequestException, UnicodeDecodeError) as e:
        # Optional: log or print the error for debugging
        print(f"Error fetching content from {url}: {e}")
        return None

In [3]:
import datetime
import os

def save_summary(summary: str, source:str) -> str:
    """
    Save a generated summary to a text file.
    Args:
        summary (str): The summary content generated by the agent.
        source (str): A short identifier or description of the source material (e.g., document title, url, or query text string). 
    Returns:
        str: The full path of the saved summary file.
    Behavior:
        - Ensures that a 'summaries' directory exists before saving.
        - Write the summary in a readable, structured format.
        - Ensure that the timestamp is included in the filename to be saved.
    Example:
    >>> save_summary("AI agents are designed to assist users...", "OpenAI_Docs")
        'summaries/OpenAI_Docs_20251026_024454.txt'
    Notes:
        - The function removes spaces and special characters from the source 
          to create a safe filename.
    """
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f'{source}_{timestamp}.txt'

    os.makedirs('summaries', exist_ok=True)
    filepath = os.path.join('summaries',filename)

    with open(filepath, 'w', encoding='utf-8') as f:
        f.write(f'Source: {source}\n\n')
        f.write(summary)
    return filepath

In [4]:
from minsearch import AppendableIndex
from typing import Any, Dict, List, Optional

class SearchTools:

    def __init__(self) -> None:
        self.index = AppendableIndex(text_fields=['content'])

    def search(self, query: str) -> List[Dict[str, Any]]:
        """
        Search the index for documents matching a query string.

        Args:
            query (str): The search query.

        Returns:
            List[Dict[str, Any]]: A list of search result dictionaries.
        """
        return self.index.search(query, num_results=5)

    def index_content(self, url: str, content: Optional[str] = None) -> str:
        """
        Index text content from a given source into the search index.

        Args:
            url (str): link to the Wikipedia page.
            content (str): Markdown content of a Wikipedia page or None if the transcript needs to be downloaded

        Returns:
            str: "SUCCESS" upon successful indexing.
        """
        if content is None:
            content = get_page_content(url)
        chunks = sliding_window(content, size=3000, step=1500)

        for chunk in chunks:
            chunk["wiki_link"] = url
            self.index.append(chunk)

        return "SUCCESS"


In [5]:
from toyaikit.llm import OpenAIClient
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner
from toyaikit.chat.runners import DisplayingRunnerCallback
from toyaikit.tools import Tools

In [6]:
from agents import Agent, function_tool
search_tools = SearchTools()

tool_methods = [
    function_tool(get_page_content),
    function_tool(save_summary),
    function_tool(search_tools.search),
    function_tool(search_tools.index_content)
]

agent_instructions = """
You are a helpful assistant who collects Markdown content from Wikipedia pages.
Once you fetch the data:
1. Ask whether the user wants a summary of the Wikipedia page to be saved. If the user indicates 'yes', invoke the 'save_summary' tool.
2. Automatically index the Markdown content for later search queries.
If the user asks a question using natural language regarding the Wikipedia page, perform a search using the `search` tool. 
If you cannot find the answer in the index, invoke the `get_page_content` tool to search Wikipedia.

You have access to the following tools:
1. get_page_content - Use this to fetch Markdown content using the Wikipedia link provided.
2. save_summary - Use this tool to save the summary to the Markdown content you generated. 
3. search - There are two parts to this tool:
    - First call `index_content` to index the webpage, and once you finish with indexing, notify the user that indexing is complete. Ask how the user wants to proceed.
    - If the user asks a natural language question, proceed to invoke the `search` tool to search for information relevant to the question.

When answering a question: 
1. Provide file references for all source materials. 
2. Be concise, accurate, and helpful — focus on clarity and usability for developers.
3. If documentation is missing or unclear, infer from context and note that explicitly.
4. If you use one of the following tools, ensure that you include why you chose to invoke it. 
""".strip()

assistant = Agent(
    name='assistant',
    tools=tool_methods,
    instructions=agent_instructions,
    model='gpt-4o-mini'
)

In [ ]:
# Set up the runner:
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIAgentsSDKRunner

chat_interface = IPythonChatInterface()

runner = OpenAIAgentsSDKRunner(
    chat_interface=chat_interface,
    agent=assistant
)

# Test
await runner.run();

You: cabybara


You: Index the following pages: - Lesser capybara — https://en.wikipedia.org/wiki/Lesser_capybara  - Hydrochoerus (genus) — https://en.wikipedia.org/wiki/Hydrochoerus  - Neochoerus (extinct genus related to capybaras) — https://en.wikipedia.org/wiki/Neochoerus  - Caviodon (extinct genus of rodents related to capybaras) — https://en.wikipedia.org/wiki/Caviodon  - Neochoerus aesopi (extinct species close to capybaras) — https://en.wikipedia.org/wiki/Neochoerus_aesopi


You: What are threats to capybara populations?
